# Prepare the open source dataset for JadBIO 

### Steps:
### 1. Load molecules + odor labels via Pyrfume
### 2. Map 138 odor labels into the dictionary 
### 3. Assign each molecule to its metacategories
### 4. Compute molecular descriptors (MACCS+ Morgan + Mordred)
### 5. Save clean CSV ready

In [1]:
import pandas as pd
import numpy as np
   
from rdkit import Chem
from rdkit.Chem import MACCSkeys, AllChem , rdFingerprintGenerator
from mordred import Calculator, descriptors


In [2]:

INput_FILE     = r"C:\Users\hp\Desktop\stage etis\open_dataset\Multi-Labelled_Smiles_Odors_dataset.csv"
OUTPUT_FILE    = "dataset_A_radius_3.csv"
CORR_THRESHOLD = 0.95

## Loading the dataset 

In [3]:
df_raw = pd.read_csv(INput_FILE)

# Na5oun les 138 columns d'odeurs
odor_cols = df_raw.columns[2:].tolist()  
behavior  = df_raw[odor_cols].copy()
behavior.index = df_raw.index

# Molecules with SMILES
molecules = df_raw[['nonStereoSMILES']].rename(columns={'nonStereoSMILES': 'SMILES'})

print('Dataset shape :', df_raw.shape)
print('Odor labels   :', len(odor_cols))
print(df_raw.head(3))

Dataset shape : (4983, 140)
Odor labels   : 138
    nonStereoSMILES                                     descriptors  \
0           CC(O)CN                                           fishy   
1     CCC(=O)C(=O)O          fatty;lactonic;sweet;caramellic;creamy   
2  O=C(O)CCc1ccccc1  rose;floral;fatty;sweet;musk;cinnamon;balsamic   

   alcoholic  aldehydic  alliaceous  almond  amber  animal  anisic  apple  \
0          0          0           0       0      0       0       0      0   
1          0          0           0       0      0       0       0      0   
2          0          0           0       0      0       0       0      0   

   ...  tropical  vanilla  vegetable  vetiver  violet  warm  waxy  weedy  \
0  ...         0        0          0        0       0     0     0      0   
1  ...         0        0          0        0       0     0     0      0   
2  ...         0        0          0        0       0     0     0      0   

   winey  woody  
0      0      0  
1      0      0  

## Mapping the 138 descriptions of the odors into 12 clusters 

In [4]:
# Dictionary of the clusters and their corresponding labels
META_CATEGORIES = {
    'floral': [
        'rose', 'jasmin', 'lily', 'muguet', 'violet', 'hyacinth',
        'geranium', 'lavender', 'orangeflower', 'chamomile', 'hawthorn'
    ],
    'fruity': [
        'apple', 'apricot', 'banana', 'berry', 'cherry', 'grape',
        'grapefruit', 'lemon', 'melon', 'orange', 'peach', 'pear', 'pineapple',
        'plum', 'raspberry', 'strawberry', 'tropical', 'black currant', 'fruit skin',
        'juicy',   'ripe',    
    ],
    'sweet': [
        'vanilla', 'caramellic', 'honey', 'chocolate', 'cocoa',
        'coconut', 'creamy', 'buttery', 'milky', 'dairy'
    ],
    'woody': [
        'cedar', 'sandalwood', 'pine', 'vetiver', 'terpenic',
        'balsamic', 'cortex','dry',     
    ],
    'green': [
        'grassy', 'herbal', 'leafy', 'hay', 'tea', 'fresh',
        'cucumber', 'vegetable', 'weedy','natural', 
    ],
    'spicy': [
        'cinnamon', 'clove', 'warm', 'pungent', 'sharp',
        'cooling', 'mint', 'camphoreous','aromatic', 
    ],
    'animal_musk': [
        'animal', 'musk', 'leathery', 'fishy', 'sweaty', 'meaty',
        'beefy', 'musty'
    ],
    'earthy': [
        'mushroom', 'nutty', 'hazelnut', 'roasted', 'coffee',
        'tobacco', 'smoky', 'popcorn'
    ],
    'citrus': [
        'bergamot', 'ozone', 'clean', 'soapy'
    ],
    'chemical': [
        'solvent', 'ethereal', 'metallic', 'medicinal', 'phenolic',
        'sulfurous', 'gassy', 'burnt', 'oily','bitter', 'fatty', 'odorless', 
        'sour',     
    ],
    'gourmand': [
        'almond', 'malty', 'rummy', 'brandy', 'cognac', 'winey',
        'cooked', 'potato', 'savory', 'celery', 'tomato', 'radish',
        'onion', 'garlic', 'cabbage', 'cheesy','alcoholic',  'alliaceous','fermented',  
    ],
    'powdery_amber': [
        'amber', 'powdery', 'anisic', 'coumarinic', 'orris',
        'waxy', 'aldehydic', 'ketonic', 'lactonic'
    ],
}

In [5]:
label_to_cat = {}
for cat, labels in META_CATEGORIES.items():
    for l in labels:
        label_to_cat[l] = cat

In [6]:
#verfifier that taxkled all the labels

ALL_138 = [
    'alcoholic', 'aldehydic', 'alliaceous', 'almond', 'amber', 'animal',
    'anisic', 'apple', 'apricot', 'aromatic', 'balsamic', 'banana', 'beefy',
    'bergamot', 'berry', 'bitter', 'black currant', 'brandy', 'burnt',
    'buttery', 'cabbage', 'camphoreous', 'caramellic', 'cedar', 'celery',
    'chamomile', 'cheesy', 'cherry', 'chocolate', 'cinnamon', 'citrus', 'clean',
    'clove', 'cocoa', 'coconut', 'coffee', 'cognac', 'cooked', 'cooling',
    'cortex', 'coumarinic', 'creamy', 'cucumber', 'dairy', 'dry', 'earthy',
    'ethereal', 'fatty', 'fermented', 'fishy', 'floral', 'fresh', 'fruit skin',
    'fruity', 'garlic', 'gassy', 'geranium', 'grape', 'grapefruit', 'grassy',
    'green', 'hawthorn', 'hay', 'hazelnut', 'herbal', 'honey', 'hyacinth',
    'jasmin', 'juicy', 'ketonic', 'lactonic', 'lavender', 'leafy', 'leathery',
    'lemon', 'lily', 'malty', 'meaty', 'medicinal', 'melon', 'metallic',
    'milky', 'mint', 'muguet', 'mushroom', 'musk', 'musty', 'natural', 'nutty',
    'odorless', 'oily', 'onion', 'orange', 'orangeflower', 'orris', 'ozone',
    'peach', 'pear', 'phenolic', 'pine', 'pineapple', 'plum', 'popcorn',
    'potato', 'powdery', 'pungent', 'radish', 'raspberry', 'ripe', 'roasted',
    'rose', 'rummy', 'sandalwood', 'savory', 'sharp', 'smoky', 'soapy',
    'solvent', 'sour', 'spicy', 'strawberry', 'sulfurous', 'sweaty', 'sweet',
    'tea', 'terpenic', 'tobacco', 'tomato', 'tropical', 'vanilla', 'vegetable',
    'vetiver', 'violet', 'warm', 'waxy', 'weedy', 'winey', 'woody'
]

covered     = [l for l in ALL_138 if l in label_to_cat]
not_covered = [l for l in ALL_138 if l not in label_to_cat]
print(f'Labels covered : {len(covered)}/138')
print(f'Not covered    : {not_covered}')
print(f'\nLabels per category:')
from collections import Counter
dist = Counter(label_to_cat[l] for l in covered)
for cat in META_CATEGORIES:
    print(f'  {cat:<20}: {dist.get(cat, 0)} labels')

Labels covered : 130/138
Not covered    : ['citrus', 'earthy', 'floral', 'fruity', 'green', 'spicy', 'sweet', 'woody']

Labels per category:
  floral              : 11 labels
  fruity              : 21 labels
  sweet               : 10 labels
  woody               : 8 labels
  green               : 10 labels
  spicy               : 9 labels
  animal_musk         : 8 labels
  earthy              : 8 labels
  citrus              : 4 labels
  chemical            : 13 labels
  gourmand            : 19 labels
  powdery_amber       : 9 labels


In [7]:

mapped_labels = [l for l in behavior.columns if l in label_to_cat]
print(f'Odor labels with a meta-category mapping: {len(mapped_labels)}')


# Compress the 138 bit label vector into a 12 bit category vector
categories = list(META_CATEGORIES.keys())
cat_binary = pd.DataFrame(0, index=behavior.index, columns=categories)

for label in mapped_labels:
    cat = label_to_cat[label]
    cat_binary[cat] = (cat_binary[cat] | behavior[label].fillna(0).astype(int))

print('\nMolecule × category matrix shape:', cat_binary.shape)
print('\nMolecules per category (positive samples per Jad Bio model):')
print(cat_binary.sum())
print('\nSample — first 5 molecules:')
print(cat_binary.head())

Odor labels with a meta-category mapping: 130

Molecule × category matrix shape: (4983, 12)

Molecules per category (positive samples per Jad Bio model):
floral            728
fruity           1384
sweet             914
woody             659
green            1556
spicy             801
animal_musk       772
earthy            808
citrus            201
chemical         2146
gourmand         1206
powdery_amber    1026
dtype: int64

Sample — first 5 molecules:
   floral  fruity  sweet  woody  green  spicy  animal_musk  earthy  citrus  \
0       0       0      0      0      0      0            1       0       0   
1       0       0      1      0      0      0            0       0       0   
2       1       0      0      1      0      1            1       0       0   
3       0       0      1      0      0      0            0       1       0   
4       0       0      1      1      1      0            0       1       0   

   chemical  gourmand  powdery_amber  
0         0         0           

### Get the smile with the 12 bit vector of corresponding categories

In [8]:

df = molecules.copy()
df.columns = ['SMILES']
df = df.join(cat_binary, how='inner')

df = df.drop_duplicates(subset='SMILES').reset_index(drop=True)

print(f'Molecules with SMILES + at least one category: {len(df)}')
print(df[['SMILES'] + categories].head(5))

Molecules with SMILES + at least one category: 4983
             SMILES  floral  fruity  sweet  woody  green  spicy  animal_musk  \
0           CC(O)CN       0       0      0      0      0      0            1   
1     CCC(=O)C(=O)O       0       0      1      0      0      0            0   
2  O=C(O)CCc1ccccc1       1       0      0      1      0      1            1   
3     OCc1ccc(O)cc1       0       0      1      0      0      0            0   
4    O=Cc1ccc(O)cc1       0       0      1      1      1      0            0   

   earthy  citrus  chemical  gourmand  powdery_amber  
0       0       0         0         0              0  
1       0       0         1         0              1  
2       0       0         1         0              0  
3       1       0         1         1              0  
4       1       0         1         1              0  


## Generating features

### MACCS fingerprint

In [9]:
def smiles_to_maccs(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = MACCSkeys.GenMACCSKeys(mol)
    return list(fp)

print('Computing MACCS fingerprints...')
maccs_rows = []
failed     = []

for _, row in df.iterrows():
    bits = smiles_to_maccs(row['SMILES'])
    if bits is not None:
        entry = {'SMILES': row['SMILES']}
        entry.update({f'MACCS_{i:03d}': bits[i] for i in range(1, 167)})
        maccs_rows.append(entry)
    else:
        failed.append(row['SMILES'])

maccs_df = pd.DataFrame(maccs_rows)
print(f'MACCS done: {len(maccs_df)} molecules × {len(maccs_df.columns)-1} bits')
if failed:
    print(f'Failed: {len(failed)} SMILES')

Computing MACCS fingerprints...
MACCS done: 4983 molecules × 166 bits


###  Morgan Fingerprints 

In [10]:

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=512)

def smiles_to_morgan(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = morgan_gen.GetCountFingerprintAsNumPy(mol)
    return fp.tolist()




print('Computing Morgan fingerprints (radius=3, 512 bits)...')
morgan_rows = []
failed_m    = []

for _, row in df.iterrows():
    bits = smiles_to_morgan(row['SMILES'])
    if bits is not None:
        entry = {'SMILES': row['SMILES']}
        entry.update({f'morgan_{i}': bits[i] for i in range(512)})
        morgan_rows.append(entry)
    else:
        failed_m.append(row['SMILES'])

morgan_df = pd.DataFrame(morgan_rows)
print(f'Morgan done: {len(morgan_df)} molecules × {len(morgan_df.columns)-1} bits')
if failed_m:
    print(f'Failed: {len(failed_m)} SMILES')

Computing Morgan fingerprints (radius=3, 512 bits)...
Morgan done: 4983 molecules × 512 bits


### Mordred Descriptors

In [11]:
def smiles_to_3d_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    if AllChem.EmbedMolecule(mol, AllChem.ETKDGv3()) != 0:
        return None
    AllChem.MMFFOptimizeMolecule(mol)
    return Chem.RemoveHs(mol)

print('Computing Mordred descriptors (this takes a few minutes)...')
calc         = Calculator(descriptors, ignore_3D=False)
mordred_rows = []

for _, row in df.iterrows():
    mol = smiles_to_3d_mol(row['SMILES'])
    if mol is None:
        print(f"  WARNING: 3D embedding failed → {row['SMILES']}")
        mordred_rows.append({'SMILES': row['SMILES']})
        continue
    desc_dict           = {str(k): v for k, v in calc(mol).items()}
    desc_dict['SMILES'] = row['SMILES']
    mordred_rows.append(desc_dict)

mordred_df            = pd.DataFrame(mordred_rows)
desc_cols             = [c for c in mordred_df.columns if c != 'SMILES']
mordred_df[desc_cols] = mordred_df[desc_cols].apply(pd.to_numeric, errors='coerce')
print(f'Raw Mordred descriptors: {len(desc_cols)}')

Computing Mordred descriptors (this takes a few minutes)...


[13:32:50] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[13:33:13] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[13:33:15] UFFTYPER: Unrecognized atom type: Ca+2 (0)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
[13:33:16] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[13:33:28] UFFTYPER: Warning: hybridization set to SP3 for atom 0
[13:33:28] UFFTYPER: Unrecognized charge state for atom: 0
[13:33:40] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[13:33:40] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[13:33:40] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[13:33:41] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[13:33:43] UFFTYPER: Unrecognized atom type: Fe2+2 (0)
[13:33:43] UFFTYPER: Unrecognized atom type: Fe2+2 (0)
[13:33:43] UFFTYPER: Unrecognized atom type: Fe2+2 (0)
[13:33

[13:46:22] UFFTYPER: Warning: hybridization set to SP3 for atom 0
[13:46:22] UFFTYPER: Unrecognized charge state for atom: 0
[13:46:29] UFFTYPER: Unrecognized atom type: Cr1+3 (0)
[13:48:28] UFFTYPER: Warning: hybridization set to SP3 for atom 0
[13:48:28] UFFTYPER: Unrecognized charge state for atom: 0
[13:49:06] UFFTYPER: Unrecognized atom type: Cr2+3 (1)
[13:49:06] UFFTYPER: Unrecognized atom type: Cr2+3 (3)
[13:50:51] UFFTYPER: Warning: hybridization set to SP3 for atom 0
[13:50:51] UFFTYPER: Unrecognized charge state for atom: 0
[13:51:15] UFFTYPER: Warning: hybridization set to SP3 for atom 0
[13:51:15] UFFTYPER: Unrecognized charge state for atom: 0
[13:51:35] UFFTYPER: Warning: hybridization set to SP3 for atom 0
[13:51:35] UFFTYPER: Unrecognized charge state for atom: 0
[13:51:35] UFFTYPER: Warning: hybridization set to SP3 for atom 0
[13:51:35] UFFTYPER: Unrecognized charge state for atom: 0
[13:53:27] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[13:53:29] UFFTYPER: Warning: h

[14:00:54] UFFTYPER: Warning: hybridization set to SP3 for atom 1
[14:00:55] UFFTYPER: Warning: hybridization set to SP3 for atom 0
[14:00:55] UFFTYPER: Unrecognized charge state for atom: 0
[14:00:59] UFFTYPER: Unrecognized charge state for atom: 1
[14:01:13] UFFTYPER: Warning: hybridization set to SP3 for atom 1
[14:01:13] UFFTYPER: Warning: hybridization set to SP3 for atom 1
[14:01:13] UFFTYPER: Warning: hybridization set to SP3 for atom 0
[14:01:13] UFFTYPER: Unrecognized charge state for atom: 0
[14:02:06] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[14:02:13] UFFTYPER: Unrecognized atom type: Co3+3 (0)
[14:02:27] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[14:02:28] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[14:02:49] UFFTYPER: Unrecognized charge state for atom: 0
[14:02:49] UFFTYPER: Unrecognized atom type: Zn+2 (0)
[14:03:05] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[14:03:05] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[14:03:06] UFFTYPER: Unrecognized atom type: Ca+2 (0)


Raw Mordred descriptors: 1826


In [12]:
mordred_df

,ABC,ABCGG,nAcid,nBase,SpAbs_A,SpMax_A,SpDiam_A,SpAD_A,SpMAD_A,LogEE_A,...,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,SMILES
0,NaN,NaN,0.0,1.0,5.226252,1.847759,3.695518,5.226252,1.045250,2.408576,...,27.254130,75.068414,5.362030,1.800000e+01,2.0,16.0,14.0,3.361111,1.333333,CC(O)CN
1,NaN,NaN,1.0,0.0,7.662988,2.052881,4.105762,7.662988,1.094713,2.766317,...,32.688753,102.031694,7.848592,4.600000e+01,6.0,26.0,26.0,4.472222,1.777778,CCC(=O)C(=O)O
2,NaN,NaN,1.0,0.0,13.674401,2.166455,4.332909,13.674401,1.243127,3.277938,...,39.846691,150.068080,7.146099,1.740000e+02,10.0,48.0,50.0,3.972222,2.583333,O=C(O)CCc1ccccc1
3,NaN,NaN,0.0,0.0,11.189957,2.193993,4.387987,11.189957,1.243329,3.089765,...,37.289972,124.052429,7.297202,9.000000e+01,9.0,40.0,43.0,3.472222,2.166667,OCc1ccc(O)cc1
4,NaN,NaN,0.0,0.0,11.189957,2.193993,4.387987,11.189957,1.243329,3.089765,...,37.289972,122.036779,8.135785,9.000000e+01,9.0,40.0,43.0,3.472222,2.166667,O=Cc1ccc(O)cc1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4978,NaN,NaN,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,74.906829,385.994041,11.028401,2.300001e+09,36.0,126.0,150.0,NaN,4.819444,O=c1[nH]cnc2c1ncn2C1OC(COP(=O)([O-])[O-])C(O)C...
4979,NaN,NaN,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,76.159956,401.004940,10.837971,2.400001e+09,38.0,132.0,157.0,NaN,4.986111,Nc1nc2c(ncn2C2OC(COP(=O)([O-])[O-])C(O)C2O)c(=...
4980,NaN,NaN,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,77.159956,438.969762,11.551836,4.900001e+09,38.0,132.0,157.0,NaN,4.986111,Nc1nc2c(ncn2C2OC(COP(=O)([O-])[O-])C(O)C2O)c(=...
4981,NaN,NaN,0.0,0.0,11.849407,2.255245,4.510490,11.849407,1.184941,3.197264,...,39.200921,175.048941,7.610824,1.160000e+02,11.0,46.0,50.0,4.333333,2.333333,CCC1SC(C)=NC(C)S1


In [13]:
#dropping molecules with all 3D descriptors NaN cuz i need to train my model on the 3d aspect 

desc_3d = [c for c in desc_cols if any(c.startswith(p) for p in 
           ['Mor', 'RDF', 'WHIM', 'GETAWAY', 'PBF', 'Autocorr3D', 
            'ESpm', 'ETA', 'IC', 'MOMI', 'GravitationalIndex'])]

print(f"3D descriptor columns identified: {len(desc_3d)}")


failed_mask = mordred_df[desc_3d].isna().all(axis=1)
print(f"Molecules with all 3D descriptors NaN so dropped : {failed_mask.sum()}")
print(f"Molecules kept: {(~failed_mask).sum()}")

mordred_df = mordred_df[~failed_mask].reset_index(drop=True)

3D descriptor columns identified: 201
Molecules with all 3D descriptors NaN so dropped : 5
Molecules kept: 4978


### Drop features the has at most 1 NAN

In [14]:
# Drop any descriptor column that has at least 1 NaN 
nan_counts = mordred_df[desc_cols].isna().sum()
keep       = nan_counts[nan_counts == 0].index.tolist()
print(f'Dropped (at least 1 NaN): {len(desc_cols) - len(keep)}')
print(f'Remaining descriptors   : {len(keep)}')
desc_cols  = keep

Dropped (at least 1 NaN): 1123
Remaining descriptors   : 703


### Drop 0 variance 

In [15]:
variance  = mordred_df[desc_cols].var()
keep      = variance[variance > 0].index.tolist()
print(f'Dropped (zero variance): {len(desc_cols) - len(keep)}')
desc_cols = keep

Dropped (zero variance): 127


### Drop highly correlated pairs 

In [16]:
corr_matrix   = mordred_df[desc_cols].corr().abs()
upper         = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop       = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]
desc_cols     = [c for c in desc_cols if c not in to_drop]
print(f'Dropped (corr > {CORR_THRESHOLD}): {len(to_drop)}')
print(f'Final Mordred descriptor count: {len(desc_cols)}')
mordred_clean = mordred_df[['SMILES'] + desc_cols]

Dropped (corr > 0.95): 249
Final Mordred descriptor count: 327


## Merge all features 

In [17]:
result_df = df[['SMILES'] + categories].copy()
result_df = result_df.merge(maccs_df,      on='SMILES', how='left')
result_df = result_df.merge(morgan_df,     on='SMILES', how='left')
result_df = result_df.merge(mordred_clean, on='SMILES', how='left')
result_df = result_df.dropna(subset=['MACCS_001'])

result_df.to_csv(OUTPUT_FILE, index=False, sep=';')

print(f'\n=== Final dataset ===')
print(f'Rows     : {len(result_df)}')
print(f'Columns  : {len(result_df.columns)}')
print(f'  → 12 binary category labels')
print(f'  → MACCS bits         : 166')
print(f'  → Morgan bits        : 512')
print(f'  → Mordred descriptors: {len(desc_cols)}')
print(f'\nPositive samples per category (for Jad Bio):')
print(result_df[categories].sum().sort_values(ascending=False))
print(f'\nSaved to: {OUTPUT_FILE}')


=== Final dataset ===
Rows     : 4983
Columns  : 1018
  → 12 binary category labels
  → MACCS bits         : 166
  → Morgan bits        : 512
  → Mordred descriptors: 327

Positive samples per category (for Jad Bio):
chemical         2146
green            1556
fruity           1384
gourmand         1206
powdery_amber    1026
sweet             914
earthy            808
spicy             801
animal_musk       772
floral            728
woody             659
citrus            201
dtype: int64

Saved to: dataset_A_radius_3.csv


## Remove the 6 molecules we have from the training dataset 


In [18]:
your_6 = {
    'CC(C)/C=C/C=C(\C)/C=C' : 'Ocimene',
    'CC1=CCC2C(C1)C2(C)C'   : 'Delta-3-Carene',
    'CC(=CCCC(C)(C=C)O)C'   : 'Linalool',
    'CC1=CCC2CC1C2(C)C'     : 'Alpha-Pinene',
    'CC1=CC[C@H](CC1)C(=C)C': 'S-Limonene',
    'CC1=CC[C@@H](CC1)C(=C)C':'R-Limonene',
}

found = result_df[result_df['SMILES'].isin(your_6.keys())].copy()
found['molecule'] = found['SMILES'].map(your_6)

print(f'Terpenes found in dataset: {len(found)}')
if len(found) > 0:
    print(found[['molecule'] + categories])
   
else:
    print('None found — SMILES may differ slightly. Check by molecule name.')

Terpenes found in dataset: 2
           molecule  floral  fruity  sweet  woody  green  spicy  animal_musk  \
129    Alpha-Pinene       0       0      0      1      1      1            0   
803  Delta-3-Carene       0       0      0      1      1      0            0   

     earthy  citrus  chemical  gourmand  powdery_amber  
129       0       0         0         0              0  
803       0       0         1         0              0  


In [19]:
result_df = result_df[~result_df['SMILES'].isin(your_6.keys())].reset_index(drop=True)
print(f"Dataset after removing terpenes: {len(result_df)} molecules")

result_df.to_csv(OUTPUT_FILE, index=False, sep=';')
print(f"Training set saved to: {OUTPUT_FILE}")

Dataset after removing terpenes: 4981 molecules
Training set saved to: dataset_A_radius_3.csv
